# 01 — LeNet → AlexNet → VGGNet: Architecture Evolution
**Phase 3 · Week 10 · Monday · Jun 8 2026 · Remote**

## What We Cover Today
| Era | Architecture | Year | Breakthrough |
|---|---|---|---|
| Classic | **LeNet-5** | 1998 | First practical CNN for digit recognition |
| Deep Learning Era | **AlexNet** | 2012 | GPU training, ReLU, Dropout, ImageNet winner |
| Very Deep | **VGGNet** | 2014 | Only 3×3 convs, depth is all you need |

**Key design lesson:** Each architecture answered a specific problem. Understanding the problem drives the solution.

---

## 1 · LeNet-5 (1998) — The Pioneer

LeNet-5 by Yann LeCun was the first successful CNN, designed for MNIST digit recognition.

**Architecture:** Input(1×32×32) → Conv(6,5×5) → AvgPool → Conv(16,5×5) → AvgPool → FC(120) → FC(84) → FC(10)

**What it proved:** Local feature detection via convolution works far better than fully-connected networks for images.

**Limitations:** Too small for large images; Sigmoid activations → vanishing gradients; too slow on CPU.

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader
import numpy as np
import matplotlib.pyplot as plt

torch.manual_seed(0)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

class LeNet5(nn.Module):
    """Original LeNet-5 (adapted: ReLU instead of Sigmoid, MaxPool instead of AvgPool)"""
    def __init__(self, in_channels=1, num_classes=10):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(in_channels, 6, kernel_size=5, padding=2),  # 28→28
            nn.ReLU(), nn.MaxPool2d(2, 2),               # 28→14
            nn.Conv2d(6, 16, kernel_size=5),              # 14→10
            nn.ReLU(), nn.MaxPool2d(2, 2),
            nn.AdaptiveAvgPool2d(4),  # fixed 4x4 regardless of input
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(16*4*4, 120), nn.ReLU(),
            nn.Linear(120, 84),     nn.ReLU(),
            nn.Linear(84, num_classes),
        )
    def forward(self, x): return self.classifier(self.features(x))

lenet = LeNet5()
x = torch.randn(1, 1, 28, 28)
print("LeNet-5 shape trace:")
for name, layer in lenet.features.named_children():
    x = layer(x)
    if hasattr(layer, 'weight') or 'Pool' in type(layer).__name__:
        print(f"  {type(layer).__name__:20s}: {x.shape}")

total_lenet = sum(p.numel() for p in lenet.parameters())
print(f"\nLeNet-5 total parameters: {total_lenet:,}")

---

## 2 · AlexNet (2012) — The Deep Learning Moment

AlexNet won ImageNet LSVRC 2012 with **top-5 error 15.3%** vs runner-up 26.2% — a shocking gap.

**What was new:**
| Innovation | Why it mattered |
|---|---|
| **ReLU activations** | 6× faster training than Sigmoid/Tanh — no vanishing gradient |
| **GPU training** | Ran on 2× GTX 580, making deep training practical |
| **Dropout (p=0.5)** | First use of Dropout as regulariser in a major net |
| **Data augmentation** | Random crops + horizontal flips → halved overfitting |
| **Local Response Normalisation** | Lateral inhibition (superseded by BatchNorm) |
| **Overlapping MaxPool** | stride < kernel size — slightly better results |

**Architecture:** 5 conv layers + 3 FC layers, 60M parameters, input 224×224×3

In [ ]:
class AlexNet(nn.Module):
    """Simplified AlexNet (CIFAR-adapted: 32×32 input instead of 224×224)"""
    def __init__(self, num_classes=10):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 64, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2, 2),   # 32→16
            nn.Conv2d(64,192, 3, padding=1), nn.ReLU(),                       # 16→16
            nn.Conv2d(192,384, 3, padding=1), nn.ReLU(),                      # 16→16
            nn.Conv2d(384,256, 3, padding=1), nn.ReLU(),                      # 16→16
            nn.Conv2d(256,256, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2, 2),  # 16→8
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Dropout(0.5), nn.Linear(256*8*8, 1024), nn.ReLU(),
            nn.Dropout(0.5), nn.Linear(1024, 512),     nn.ReLU(),
            nn.Linear(512, num_classes),
        )
    def forward(self, x): return self.classifier(self.features(x))

alexnet = AlexNet()
total_alex = sum(p.numel() for p in alexnet.parameters())
print(f"AlexNet (CIFAR-adapted) parameters: {total_alex:,}")

# Architecture comparison table
print("\nArchitecture Comparison:")
print(f"{'Model':12} {'Year':6} {'Params':>12} {'Depth':>7} {'Key Innovation'}")
print("-" * 65)
print(f"{'LeNet-5':12} {'1998':6} {total_lenet:>12,} {'5':>7}  First practical CNN")
print(f"{'AlexNet':12} {'2012':6} {total_alex:>12,} {'8':>7}  ReLU + Dropout + GPU")
print(f"{'VGGNet-16':12} {'2014':6} {'138,357,544':>12} {'16':>7}  Deep 3×3 only stacks")
print(f"{'ResNet-50':12} {'2015':6} {'25,557,032':>12} {'50':>7}  Skip connections")

---

## 3 · VGGNet (2014) — Depth With Simplicity

Oxford's VGG group asked: **what if we just make it deeper, using only 3×3 convolutions?**

**The 3×3 insight:**
- Two stacked 3×3 convs have the same receptive field as one 5×5 conv
- But use **fewer parameters**: 2×(3²C²) = 18C² vs 5²C² = 25C²
- AND have an extra non-linearity between them → more discriminative

**VGG-16 structure:** 13 conv layers + 3 FC layers — all 3×3 convs, max-pool between blocks

| Block | Layers | Channels | Spatial |
|---|---|---|---|
| Block 1 | 2×Conv(3×3) | 64 | 224→112 |
| Block 2 | 2×Conv(3×3) | 128 | 112→56 |
| Block 3 | 3×Conv(3×3) | 256 | 56→28 |
| Block 4 | 3×Conv(3×3) | 512 | 28→14 |
| Block 5 | 3×Conv(3×3) | 512 | 14→7 |
| FC | 3×FC | 4096/4096/1000 | — |

In [ ]:
def make_vgg_block(in_ch, out_ch, n_convs):
    layers = []
    for _ in range(n_convs):
        layers += [nn.Conv2d(in_ch, out_ch, 3, padding=1), nn.ReLU()]
        in_ch = out_ch
    layers.append(nn.MaxPool2d(2, 2))
    return nn.Sequential(*layers)

class VGGNet(nn.Module):
    """VGG-style net (CIFAR-adapted: 32×32 input, smaller FC)"""
    def __init__(self, num_classes=10):
        super().__init__()
        self.features = nn.Sequential(
            make_vgg_block(3,  64, 2),   # 32→16
            make_vgg_block(64, 128, 2),  # 16→8
            make_vgg_block(128,256, 3),  # 8→4
        )
        self.classifier = nn.Sequential(
            nn.AdaptiveAvgPool2d(2),     # → 256×2×2 regardless of input size
            nn.Flatten(),
            nn.Linear(256*4, 512), nn.ReLU(), nn.Dropout(0.5),
            nn.Linear(512, 256),   nn.ReLU(), nn.Dropout(0.5),
            nn.Linear(256, num_classes),
        )
    def forward(self, x): return self.classifier(self.features(x))

vgg = VGGNet()
total_vgg = sum(p.numel() for p in vgg.parameters())
print(f"VGGNet (CIFAR-adapted) parameters: {total_vgg:,}")

# Shape trace
x = torch.randn(1, 3, 32, 32)
print("\nVGGNet shape trace:")
for i, block in enumerate(vgg.features):
    x = block(x)
    print(f"  Block {i+1}: {x.shape}")

---

## 4 · Training All Three on Synthetic CIFAR-style Data

In [ ]:
# Synthetic 3-channel 32×32 data (5 classes)
def make_cifar_loader(n=3000, n_cls=10, batch=128):
    X = torch.randn(n, 3, 32, 32)
    y = torch.randint(0, n_cls, (n,))
    # Add class-specific signal
    for cls in range(n_cls):
        mask = y == cls
        X[mask, cls % 3] += 1.2
    return DataLoader(TensorDataset(X, y), batch_size=batch, shuffle=True)

loader   = make_cifar_loader(n=4000, n_cls=10)
val_data = next(iter(make_cifar_loader(n=800, n_cls=10, batch=800)))
Xv, yv   = val_data[0].to(device), val_data[1].to(device)

def train_quick(model, loader, n_epochs=15, lr=1e-3):
    model = model.to(device)
    opt   = optim.Adam(model.parameters(), lr=lr)
    crit  = nn.CrossEntropyLoss()
    losses = []
    for _ in range(n_epochs):
        model.train(); ep = []
        for Xb, yb in loader:
            Xb, yb = Xb.to(device), yb.to(device)
            opt.zero_grad()
            loss = crit(model(Xb), yb)
            loss.backward(); opt.step()
            ep.append(loss.item())
        losses.append(np.mean(ep))
    model.eval()
    with torch.no_grad():
        acc = (model(Xv).argmax(1) == yv).float().mean().item()
    return losses, acc

print("Training 3 architectures (15 epochs each)...")
results = {}
for name, arch in [('LeNet-5', LeNet5(in_channels=3, num_classes=10)),
                   ('AlexNet', AlexNet(num_classes=10)),
                   ('VGGNet',  VGGNet(num_classes=10))]:
    losses, acc = train_quick(arch, loader)
    results[name] = (losses, acc)
    print(f"  {name:10s} — final loss: {losses[-1]:.4f}  val acc: {acc:.3f}")

fig, ax = plt.subplots(figsize=(10, 4))
colors = ['coral', 'steelblue', 'seagreen']
for (name, (losses, acc)), color in zip(results.items(), colors):
    ax.plot(losses, lw=2.5, label=f"{name} (acc={acc:.2f})", color=color)
ax.set_title("Loss Curves: LeNet vs AlexNet vs VGGNet", fontweight='bold')
ax.set_xlabel("Epoch"); ax.set_ylabel("Loss"); ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig("cnn_architecture_comparison.png", dpi=120, bbox_inches='tight')
plt.show()

---

## What I Learned Today

**LeNet-5** — Proved CNNs work for images. Small network, sigmoid activations, CPU-only. Foundation for everything that followed.

**AlexNet** — The moment deep learning became mainstream. Three key innovations made it work: ReLU (fast gradients), Dropout (regularisation), GPU training (scale). Won ImageNet by a margin that shocked everyone.

**VGGNet** — Showed that depth beats complexity. Using only 3×3 convolutions in deep stacks achieves the same receptive fields as larger kernels but with fewer parameters and more non-linearities. Simple, uniform design.

**3×3 stack insight:** Two 3×3 convs = one 5×5 conv in receptive field, but fewer params (18C² vs 25C²). Three 3×3 convs = one 7×7 conv (27C² vs 49C²). Always prefer deep stacks of small kernels.

**Evolution summary:**
```
1998 LeNet   — CNNs work for images (5 layers, 60k params)
2012 AlexNet — GPU + ReLU + Dropout → scale to ImageNet (8 layers, 60M params)
2014 VGGNet  — Depth with 3×3 only → better accuracy (16 layers, 138M params)
2015 ResNet  — Skip connections → depth without vanishing gradient (50-152 layers)
```